<div style="background-color:#EAF4EC; padding:20px; border-radius:10px;">

<h2 style="color:#2F6F4E; text-align:center; margin-bottom:5px;">
Forecasting: Econometric Baseline — Fixed Effects Panel Regression
</h2>

<h4 style="color:#2F6F4E; text-align:center; margin-top:0;">
Master Thesis – ESG Governance Indicators (EU-27)
</h4>

<p style="font-size:14px; color:#2F6F4E;">
This notebook implements the <strong>econometric baseline</strong> for the forecasting stage of the
CRISP-ML(Q) methodology. The objective is to estimate a <strong>Fixed Effects panel regression model</strong>
for each ESG governance indicator across the EU-27.
</p>

<p style="font-size:14px; color:#2F6F4E;">
The results obtained in this notebook serve as a <strong>reference benchmark</strong> against which the performance
of non-linear machine learning models (e.g., Random Forest, XGBoost, ANN, LSTM) is assessed in later stages
of the analysis.
</p>

</div>

<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;"> <h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;"> 1. Data Integrity — Do we have the dataset we expect? </h2> </div>

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import r2_score
from linearmodels.panel import PanelOLS, PooledOLS, RandomEffects
import statsmodels.api as sm

df_wide = pd.read_csv("../data/processed/data_final_EU27.csv")

In [2]:
# % de NaNs por coluna
nan_pct = df_wide.isnull().mean().sort_values(ascending=False) * 100

print("NaNs por coluna (%)")
print(nan_pct[nan_pct > 0].round(2))

print(f"\n Shape original: {df_wide.shape} ")
print(f"Linhas sem nenhum NaN: {df_wide.dropna().shape[0]} ")
print(f"Linhas perdidas com dropna: {df_wide.shape[0] - df_wide.dropna().shape[0]} ")

NaNs por coluna (%)
Series([], dtype: float64)

 Shape original: (405, 28) 
Linhas sem nenhum NaN: 405 
Linhas perdidas com dropna: 0 


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;"> <h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;"> 2. Temporal Structure — Identifying the available time dimension </h2> </div>

In [3]:
# 1) identificar colunas de anos (strings "2000", "2001", ...)
year_cols = [c for c in df_wide.columns if str(c).isdigit()]
print("Years:", year_cols[:5], "...", year_cols[-5:])
print("N years:", len(year_cols))


Years: ['2000', '2001', '2002', '2003', '2004'] ... ['2019', '2020', '2021', '2022', '2023']
N years: 24


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;"> <h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;"> Panel Structure — Constructing a lagged country–year panel </h2> </div>

In [4]:
def prepare_panel_with_covariates(df_wide_raw: pd.DataFrame, target_code: str) -> pd.DataFrame:
    # Identificar colunas de anos
    id_cols = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]
    year_cols = [c for c in df_wide_raw.columns if c not in id_cols]

    # Pivot: índice = (Country Code, Year), colunas = Indicator Code
    df = df_wide_raw.set_index(id_cols)[year_cols]
    df = df.stack().reset_index()
    df.columns = ["Country Name", "Country Code", "Indicator Name", "Indicator Code", "Year", "Value"]
    df["Year"] = df["Year"].astype(int)

    # Pivot para wide com indicadores como colunas
    df_p = df.pivot_table(index=["Country Code", "Year"], columns="Indicator Code", values="Value").sort_index()

    # Target e lags
    df_p["y"] = df_p[target_code]
    covariates = [c for c in df_p.columns if c != target_code and c != "y"]
    df_p["y_lag1"] = df_p.groupby("Country Code")[target_code].shift(1)
    for col in covariates:
        df_p[f"{col}_lag1"] = df_p.groupby("Country Code")[col].shift(1)

    lag_cols = ["y", "y_lag1"] + [f"{c}_lag1" for c in covariates]
    return df_p[lag_cols].dropna()

panel_cc = prepare_panel_with_covariates(df_wide, "CC.EST")
print("Shape:", panel_cc.shape)
panel_cc.head()

Shape: (621, 16)


Indicator Code            y    y_lag1  GB.XPD.RSDV.GD.ZS_lag1  GE.EST_lag1  \
Country Code Year                                                            
AUT          2001  1.751059  1.751059                 1.88602     1.847265   
             2002  1.915159  1.751059                 1.99210     1.847265   
             2003  1.963869  1.915159                 2.06598     1.880228   
             2004  2.026624  1.963869                 2.17456     1.930461   
             2005  1.912423  2.026624                 2.16612     1.843771   

Indicator Code     IP.JRN.ARTC.SC_lag1  IP.PAT.RESD_lag1  IT.NET.USER.ZS_lag1  \
Country Code Year                                                               
AUT          2001              6614.60            1961.0                 33.7   
             2002              7073.06            1815.0                 39.2   
             2003              7505.60            1932.0                 36.6   
             2004              7972.47            2120.0                 42.7   
             2005              8157.43            2248.0                 54.3   

Indicator Code     NY.GDP.MKTP.KD.ZG_lag1  PV.EST_lag1  RL.EST_lag1  \
Country Code Year                                                     
AUT          2001                3.189523     0.822056     1.812333   
             2002                1.316987     0.822056     1.812333   
             2003                1.484369     1.358955     1.857913   
             2004                1.141565     0.962002     1.847917   
             2005                2.565255     1.089649     1.816664   

Indicator Code     RQ.EST_lag1  SE.ENR.PRSC.FM.ZS_lag1  SG.GEN.PARL.ZS_lag1  \
Country Code Year                                                             
AUT          2001     1.475476                 0.96969            26.775956   
             2002     1.475476                 0.96958            26.775956   
             2003     1.538056                 0.96443            26.775956   
             2004     1.537136                 0.96342            33.879781   
             2005     1.514103                 0.96304            33.879781   

Indicator Code     SL.TLF.CACT.FM.ZS_lag1  SM.POP.NETM_lag1  VA.EST_lag1  
Country Code Year                                                         
AUT          2001               71.262746           17631.0     1.317559  
             2002               72.136593           39866.0     1.317559  
             2003               73.721597           37508.0     1.307497  
             2004               73.940179           44362.0     1.343882  
             2005               76.023205           55007.0     1.475651

<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;"> <h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;"> 6. Temporal Splitting  </h2> </div>

In [5]:
def time_split(panel_df: pd.DataFrame, n_train_years=16, n_val_years=3, n_test_years=2):
    years = sorted(panel_df.index.get_level_values("Year").unique())
    total_needed = n_train_years + n_val_years + n_test_years

    if len(years) < total_needed:
        raise ValueError(
            f"Not enough years to split: have {len(years)} years, need {total_needed}."
        )

    test_years = years[-n_test_years:]
    val_years = years[-(n_test_years + n_val_years):-n_test_years]
    train_years = years[-(n_test_years + n_val_years + n_train_years):-(n_test_years + n_val_years)]

    train_df = panel_df.loc[(slice(None), train_years), :]
    val_df   = panel_df.loc[(slice(None), val_years), :]
    test_df  = panel_df.loc[(slice(None), test_years), :]

    return train_df, val_df, test_df, train_years, val_years, test_years


In [6]:
panel_cc = prepare_panel_with_covariates(df_wide, "CC.EST")  

train_df, val_df, test_df, train_years, val_years, test_years = time_split(panel_cc)

print("Train years:", train_years[0], "-", train_years[-1], "| rows:", len(train_df))
print("Val years:", val_years[0], "-", val_years[-1], "| rows:", len(val_df))
print("Test years:", test_years[0], "-", test_years[-1], "| rows:", len(test_df))

Train years: 2003 - 2018 | rows: 432
Val years: 2019 - 2021 | rows: 81
Test years: 2022 - 2023 | rows: 54


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;"> <h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;"> 7. Econometric Baseline — Fixed Effects panel regression </h2> </div>

In [7]:
def fit_fe(train_df: pd.DataFrame):
    X_train = train_df.drop(columns=["y"]).copy()
    X_train["const"] = 1.0
    y_train = train_df["y"]
    model = PanelOLS(y_train, X_train, entity_effects=True)
    res = model.fit(cov_type="clustered", cluster_entity=True)
    return res

def predict_and_metrics(res, df_part: pd.DataFrame):
    X = df_part.drop(columns=["y"]).copy()
    X["const"] = 1.0
    y_true = df_part["y"]

    train_cols = res.model.exog.vars
    X = X[train_cols]

    y_pred = res.predict(exog=X).iloc[:, 0]

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)

    return mae, rmse, r2

<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;"> <h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;"> 8. Baseline Evaluation — Loop over all indicators </h2> </div>

In [8]:
BASELINE_INDICATORS = [
     "CC.EST",
    "GB.XPD.RSDV.GD.ZS",
    "GE.EST",
    "IP.JRN.ARTC.SC",
    "IP.PAT.RESD",
    "IT.NET.USER.ZS",
    "NY.GDP.MKTP.KD.ZG",
    "PV.EST",
    "RL.EST",
    "RQ.EST",
    "SE.ENR.PRSC.FM.ZS",
    "SG.GEN.PARL.ZS",
    "SL.TLF.CACT.FM.ZS",
    "SM.POP.NETM",
    "VA.EST"
]

In [9]:
def prepare_panel_with_covariates(df_wide_raw: pd.DataFrame, target_code: str) -> pd.DataFrame:
    id_cols = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]
    year_cols = [c for c in df_wide_raw.columns if c not in id_cols]

    df = df_wide_raw.set_index(id_cols)[year_cols]
    df = df.stack().reset_index()
    df.columns = ["Country Name", "Country Code", "Indicator Name", "Indicator Code", "Year", "Value"]
    df["Year"] = df["Year"].astype(int)

    df_p = df.pivot_table(
        index=["Country Code", "Year"],
        columns="Indicator Code",
        values="Value"
    ).sort_index()

    df_p["y"] = df_p[target_code]
    covariates = [c for c in df_p.columns if c != target_code and c != "y"]

    df_p["y_lag1"] = df_p.groupby("Country Code")[target_code].shift(1)
    for col in covariates:
        df_p[f"{col}_lag1"] = df_p.groupby("Country Code")[col].shift(1)

    lag_cols = ["y", "y_lag1"] + [f"{c}_lag1" for c in covariates]
    return df_p[lag_cols].dropna()

In [10]:
def time_split(panel_df: pd.DataFrame, n_train_years=16, n_val_years=3, n_test_years=2):
    years = sorted(panel_df.index.get_level_values("Year").unique())
    total_needed = n_train_years + n_val_years + n_test_years

    if len(years) < total_needed:
        raise ValueError(f"Not enough years to split: have {len(years)}, need {total_needed}.")

    test_years  = years[-n_test_years:]
    val_years   = years[-(n_test_years + n_val_years):-n_test_years]
    train_years = years[-(n_test_years + n_val_years + n_train_years):-(n_test_years + n_val_years)]

    train_df = panel_df.loc[(slice(None), train_years), :]
    val_df   = panel_df.loc[(slice(None), val_years), :]
    test_df  = panel_df.loc[(slice(None), test_years), :]

    return train_df, val_df, test_df, train_years, val_years, test_years

In [11]:
def fit_fe(train_df: pd.DataFrame):
    X_train = train_df.drop(columns=["y"]).copy()
    X_train["const"] = 1.0
    y_train = train_df["y"]

    model = PanelOLS(y_train, X_train, entity_effects=True, drop_absorbed=True)
    res = model.fit(cov_type="clustered", cluster_entity=True)
    return res

In [12]:
baseline_results = []
baseline_models  = {}

for ind in BASELINE_INDICATORS:
    panel_df = prepare_panel_with_covariates(df_wide, ind)

    train_df, val_df, test_df, train_years, val_years, test_years = time_split(panel_df)

    res = fit_fe(train_df)

    mae_val,  rmse_val,  r2_val  = predict_and_metrics(res, val_df)
    mae_test, rmse_test, r2_test = predict_and_metrics(res, test_df)

    baseline_models[ind] = res

    baseline_results.append({
        "Indicator Code": ind,
        "MAE_Val":    mae_val,
        "RMSE_Val":   rmse_val,
        "R2_Val":     r2_val,
        "MAE_Test":   mae_test,
        "RMSE_Test":  rmse_test,
        "R2_Test":    r2_test,
        "Train_Years": f"{train_years[0]}-{train_years[-1]}",
        "Val_Years":   f"{val_years[0]}-{val_years[-1]}",
        "Test_Years":  f"{test_years[0]}-{test_years[-1]}",
        "N_Train":    len(train_df),
        "N_Val":      len(val_df),
        "N_Test":     len(test_df),
        "Countries":  panel_df.index.get_level_values("Country Code").nunique()
    })
    print(f" {ind} | MAE_Test: {mae_test:.4f} | RMSE_Test: {rmse_test:.4f} | R2_Test: {r2_test:.4f}")

 CC.EST | MAE_Test: 0.1889 | RMSE_Test: 0.2723 | R2_Test: 0.8672
 GB.XPD.RSDV.GD.ZS | MAE_Test: 0.2397 | RMSE_Test: 0.3365 | R2_Test: 0.8599
 GE.EST | MAE_Test: 0.2346 | RMSE_Test: 0.3161 | R2_Test: 0.6836
 IP.JRN.ARTC.SC | MAE_Test: 1019.7181 | RMSE_Test: 1832.2609 | R2_Test: 0.9959
 IP.PAT.RESD | MAE_Test: 1440.9355 | RMSE_Test: 2518.4945 | R2_Test: 0.8966
 IT.NET.USER.ZS | MAE_Test: 3.6613 | RMSE_Test: 5.4512 | R2_Test: -0.2478
 NY.GDP.MKTP.KD.ZG | MAE_Test: 6.4690 | RMSE_Test: 10.1557 | R2_Test: -12.9187
 PV.EST | MAE_Test: 0.1658 | RMSE_Test: 0.2257 | R2_Test: -0.0902
 RL.EST | MAE_Test: 0.1435 | RMSE_Test: 0.2545 | R2_Test: 0.7956
 RQ.EST | MAE_Test: 0.1434 | RMSE_Test: 0.2165 | R2_Test: 0.8126
 SE.ENR.PRSC.FM.ZS | MAE_Test: 0.0270 | RMSE_Test: 0.0446 | R2_Test: -1.6092
 SG.GEN.PARL.ZS | MAE_Test: 3.2330 | RMSE_Test: 4.2365 | R2_Test: 0.7890
 SL.TLF.CACT.FM.ZS | MAE_Test: 1.3065 | RMSE_Test: 1.7340 | R2_Test: 0.8975
 SM.POP.NETM | MAE_Test: 260206.1253 | RMSE_Test: 468510.1822 | 

In [13]:
baseline_results_df = pd.DataFrame(baseline_results).sort_values("RMSE_Test")
baseline_results_df

,Indicator Code,MAE_Val,RMSE_Val,R2_Val,MAE_Test,RMSE_Test,R2_Test,Train_Years,Val_Years,Test_Years,N_Train,N_Val,N_Test,Countries
10,SE.ENR.PRSC.FM.ZS,0.027468,0.049496,-2.260130,0.027032,0.044649,-1.609151,2003-2018,2019-2021,2022-2023,432,81,54,27
14,VA.EST,0.118031,0.218541,0.623384,0.122645,0.208899,0.678940,2003-2018,2019-2021,2022-2023,432,81,54,27
9,RQ.EST,0.161224,0.268308,0.671634,0.143365,0.216542,0.812630,2003-2018,2019-2021,2022-2023,432,81,54,27
7,PV.EST,0.187454,0.240996,0.159176,0.165829,0.225656,-0.090167,2003-2018,2019-2021,2022-2023,432,81,54,27
8,RL.EST,0.157006,0.287697,0.748604,0.143490,0.254486,0.795642,2003-2018,2019-2021,2022-2023,432,81,54,27
0,CC.EST,0.210008,0.294055,0.850412,0.188885,0.272289,0.867228,2003-2018,2019-2021,2022-2023,432,81,54,27
2,GE.EST,0.241778,0.341174,0.637508,0.234583,0.316088,0.683594,2003-2018,2019-2021,2022-2023,432,81,54,27
1,GB.XPD.RSDV.GD.ZS,0.248011,0.361163,0.834889,0.239747,0.336504,0.859864,2003-2018,2019-2021,2022-2023,432,81,54,27
12,SL.TLF.CACT.FM.ZS,1.068775,1.755717,0.897142,1.306474,1.733981,0.897499,2003-2018,2019-2021,2022-2023,432,81,54,27
11,SG.GEN.PARL.ZS,3.224858,4.023190,0.826072,3.232986,4.236474,0.788995,2003-2018,2019-2021,2022-2023,432,81,54,27


# Baseline Panel Regression — Results Analysis

## Temporal Split Configuration
| Period | Years | Observations |
|---|---|---|
| **Train** | 2003–2018 | 432 |
| **Validation** | 2019–2021 | 81 |
| **Test** | 2022–2023 | 54 |

> Balanced panel with **27 countries** and **15 indicators**. Year 2000 is sacrificed for lag construction. The model uses country Fixed Effects with clustered standard errors.

---

## Results by Group

### Group 1 — Strong Performance (R²_Test > 0.85)

| Indicator Code | MAE_Test | RMSE_Test | R²_Test |
|---|---|---|---|
| IP.JRN.ARTC.SC | 1019.72 | 1832.26 | **0.9959** |
| IP.PAT.RESD | 1440.94 | 2518.49 | **0.8966** |
| SL.TLF.CACT.FM.ZS | 1.31 | 1.73 | **0.8975** |
| CC.EST | 0.19 | 0.27 | **0.8672** |
| GB.XPD.RSDV.GD.ZS | 0.24 | 0.34 | **0.8599** |

The model captures the temporal dynamics of these indicators well. Typical of **persistent** series — indicators that change slowly year-on-year, where the lagged value is a strong predictor.

---

### Group 2 — Moderate Performance (R²_Test 0.65–0.85)

| Indicator Code | MAE_Test | RMSE_Test | R²_Test |
|---|---|---|---|
| RQ.EST | 0.14 | 0.22 | **0.8126** |
| RL.EST | 0.14 | 0.25 | **0.7956** |
| SG.GEN.PARL.ZS | 3.23 | 4.24 | **0.7890** |
| GE.EST | 0.23 | 0.32 | **0.6836** |
| VA.EST | 0.12 | 0.21 | **0.6789** |

Acceptable baseline performance. Room for improvement with more sophisticated forecasting models.

---

### Group 3 — Poor Performance (R²_Test < 0)

| Indicator Code | MAE_Test | RMSE_Test | R²_Test |
|---|---|---|---|
| NY.GDP.MKTP.KD.ZG | 6.47 | 10.16 | **-12.9187** |
| SM.POP.NETM | 260206.13 | 468510.18 | **-4.2826** |
| SE.ENR.PRSC.FM.ZS | 0.03 | 0.04 | **-1.6092** |
| IT.NET.USER.ZS | 3.66 | 5.45 | **-0.2478** |
| PV.EST | 0.17 | 0.23 | **-0.0902** |

A negative R² means the model performs **worse than simply predicting the mean**. Each case has a structural explanation:
- **NY.GDP.MKTP.KD.ZG** — GDP growth is highly volatile and sensitive to macroeconomic shocks not captured by lagged values
- **SM.POP.NETM** — net migration is driven by sudden geopolitical and economic events
- **SE.ENR.PRSC.FM.ZS** — school enrollment shows non-linear trends across EU countries
- **IT.NET.USER.ZS** — internet usage was in a non-linear adoption phase during the sample period
- **PV.EST** — political stability is influenced by discrete events that lags cannot anticipate

---

## Key Takeaways

1. **Fixed Effects performs well for slow-moving, persistent indicators** — governance, patents, R&D expenditure, and labour market indicators.
2. **The model struggles with volatile and event-driven indicators** — GDP growth, migration, and internet adoption.
3. **Negative R² results justify the use of more advanced models** (Random Forest, XGBoost), providing a clear motivation for moving beyond the baseline.
4. **The panel structure is balanced** — all 15 indicators cover the same 27 EU countries over the same time periods.